# **Start Section:**

In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/streamflow_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d03_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # streamflow train
    shutil.copy(glob.glob("/kaggle/input/**/d03_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # streamflow test
    print("Kaggle: streamflow data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: streamflow data copied to ./drive/MyDrive/Regression/streamflow_uncertainty_analysis/data


In [2]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks, pinned to the Colab versions,
# and do NOT restart the kernel with os._exit(0).
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 mapie==0.6.0 puncc==0.8.0 properscoring==0.1 XlsxWriter==3.2.9
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install mapie==0.6.0 -q
    !pip install puncc==0.8.0 -q
    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ngboost
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from optuna.samplers import BaseSampler
from optuna.samplers import GridSampler
from optuna.samplers import TPESampler
from optuna.samplers import PartialFixedSampler
from optuna.samplers import CmaEsSampler
from optuna.samplers import QMCSampler
from optuna.samplers import NSGAIIISampler
from optuna.samplers import NSGAIISampler
from optuna.samplers import BruteForceSampler
from optuna.samplers import GPSampler
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from openpyxl.drawing.image import Image
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from mapie.subsample import Subsample
from mapie.regression import MapieRegressor
from deel.puncc.metrics import regression_sharpness, regression_mean_coverage
from deel.puncc.api.prediction import BasePredictor, DualPredictor
from deel.puncc.regression import SplitCP, CVPlus, CQR
from deel.puncc.plotting import plot_prediction_intervals
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from mapie.metrics import regression_coverage_score
from sklearn.model_selection import KFold
from PIL import Image as PImage
from pytorch_tabnet import TabNetRegressor

[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


I0000 00:00:1795378632.047869      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1795378651.030932      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [4]:
# Go to find & replace button and replace (streamflow_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.

In [5]:
feature_names = ['pH', 'Salinity', 'Turbidity ', 'Water Temperature']
TARGET_LABEL = 'TDS'  # FIX B12: target name used in plot labels (was 'str', the concrete dataset's target, on every dataset)

In [6]:
train_data_path = "./drive/MyDrive/Regression/streamflow_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/streamflow_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")

Training data loaded successfully.
Test data loaded successfully.


In [7]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


Shape of training data: (192, 5)
First 5 rows of training data:
     pH  Salinity  Turbidity   Water Temperature  TDS
0  7.6       0.2         3.0               19.9  290
1  7.4       0.1         0.5               20.7  120
2  7.3       0.2        10.7               23.6  200
3  8.0       0.1         3.0               27.0   94
4  7.8       0.4         1.4               28.4  330

Shape of test data: (96, 5)
First 5 rows of test data:
     pH  Salinity  Turbidity   Water Temperature  TDS
0  7.2       0.1         9.1               18.8  160
1  7.4       0.2         3.7               26.3  190
2  7.0       0.1         3.0               21.6  180
3  8.1       0.1         2.2               17.2  110
4  7.9       0.1        14.1               27.0  200


In [8]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
# FIX D4 / D5 (approved 2026-11-22): settings that follow the test-set size
N_BINS = max(2, min(10, len(y_test) // 10))  # D4: bins for binned metrics (was 10 on every dataset; pile: 2-3 points per bin)
COP_T_BURNIN = min(50, len(y_test) // 2)  # D5: COP burn-in (was 50, so with < 50 test points COP's correction never started)
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (192, 4)
Shape of y_train: (192,)
Shape of X_test: (96, 4)
Shape of y_test: (96,)


In [9]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])


First five rows of normalized X_train:
[[-0.1272241   0.06262254 -0.51513341 -1.14481478]
 [-0.57957647 -0.38434874 -0.6817983  -0.97088446]
 [-0.80575265  0.06262254 -0.00180554 -0.34038706]
 [ 0.77748062 -0.38434874 -0.51513341  0.39881678]
 [ 0.32512826  0.95656508 -0.62179894  0.70319483]]

First five rows of normalized X_test:
[[-1.03192883 -0.38434874 -0.10847107 -1.38396896]
 [-0.57957647  0.06262254 -0.46846724  0.24662775]
 [-1.48428119 -0.38434874 -0.51513341 -0.77521285]
 [ 1.00365681 -0.38434874 -0.56846617 -1.73182959]
 [ 0.55130444 -0.38434874  0.22485872  0.39881678]]


# **Functions:**

In [10]:
# Define the model classes
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}

def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

In [11]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'NGBoost': 'NGBoost',
        'GradientBoosting': 'Gradient Boosting',
        'HGBR' : 'HistGradientBoosting',
        'CatBoost' : 'CatBoost',
        'TabNet' : 'TabNet',
        'PGBM' : 'PGBM'
    }

    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")

    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}

    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])

    # Return the best hyperparameters
    return best_entry_value['best_params']

In [12]:
# FIX Q2: quantile crossing. Each quantile level is a separate model, so for some rows the
# predictions can come out in the wrong order (e.g. q0.10 < q0.05). Before any metric we sort
# each row's quantile predictions (monotone rearrangement, Chernozhukov et al. 2010) and record
# how many rows crossed, for the Excel report.
quantile_crossing_report = {}

def rearrange_quantiles(predictions_df, tag=None):
    q_cols = sorted(c for c in predictions_df.columns if c != 'Actual')
    values = predictions_df[q_cols].to_numpy(dtype=float)
    crossed = (np.diff(values, axis=1) < 0).any(axis=1)
    if tag is not None:
        quantile_crossing_report[tag] = {'rows': int(len(values)), 'rows_crossing': int(crossed.sum()),
                                         'pct_crossing': round(100.0 * crossed.mean(), 2)}
    out = predictions_df.copy()
    out[q_cols] = np.sort(values, axis=1)
    return out


In [13]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def quantile_regression_print(predictions_df, file_path):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    # Extract actual values
    actual_values = predictions_df['Actual'].values

    # Debugging: Print the first few actual values
    print("First few actual values:", actual_values[:5])

    # Calculate coverage for the specific interval (0.05 to 0.95)
    lower_quantile = 0.05
    median_quantile = 0.5
    upper_quantile = 0.95

    lower_preds = predictions_df[lower_quantile].values
    median_preds = predictions_df[median_quantile].values
    upper_preds = predictions_df[upper_quantile].values

    in_interval = ((actual_values >= lower_preds) & (actual_values <= upper_preds))
    coverage = in_interval.mean()
    print(f"Coverage of 90% prediction interval: {coverage * 100:.2f}%")

    # Create a new Excel workbook and add a worksheet
    wb = Workbook()
    ws = wb.active
    ws.title = "quantile_regression_print"

    # Plot actual vs. all predicted quantiles
    num_points = len(predictions_df)
    indices = np.arange(num_points)

    plt.figure(figsize=(12, 6))
    plt.plot(indices, actual_values[:num_points], label=f'Actual {TARGET_LABEL}', marker='o', linestyle='-', color='black')

    # Loop over all columns assumed to be quantile predictions
    for column in predictions_df.columns:
        if column != 'Actual':  # Skip the actual values column
            plt.plot(indices, predictions_df[column][:num_points],
                     label=f'Predicted Quantile {column}', linestyle='--')

    plt.xlabel('Sample Number')
    plt.ylabel(f'{TARGET_LABEL}')
    plt.title(f'Actual vs. Predicted Quantile {TARGET_LABEL}')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)

    # Save the plot to a BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Insert the image into the Excel sheet
    img = Image(img_data)
    ws.add_image(img, 'A1')

    # Plot actual values and specific prediction intervals (0.05, 0.5, 0.95)
    plt.figure(figsize=(12, 6))
    plt.plot(indices, actual_values, label=f'Actual {TARGET_LABEL}', marker='o', linestyle='-', color='black')  # Connect points with a line
    plt.plot(indices, median_preds, label=f'Median Prediction ({median_quantile})', marker='x', linestyle='-', color='blue')
    plt.fill_between(
        indices,
        lower_preds,
        upper_preds,
        color='blue',
        alpha=0.5,
        label=f'Prediction Interval ({lower_quantile}-{upper_quantile})'
    )
    plt.xlabel('Sample Number')
    plt.ylabel(f' {TARGET_LABEL}')
    plt.title(f'Actual {TARGET_LABEL} with Prediction Intervals')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1, 1))

    # Save the second plot to another BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Insert the second image into the Excel sheet
    img = Image(img_data)
    ws.add_image(img, 'A30')  # Adjust the cell position as needed

    # Save the workbook
    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

In [14]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def evaluation_features_print(predictions_df, X_test, file_path):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    # Ensure X_test is a DataFrame
    if isinstance(X_test, np.ndarray):
        X_test = pd.DataFrame(X_test, columns=feature_names)
    X_test.columns = feature_names

    # Get quantile columns from predictions_df except 'Actual'
    quantile_columns = [col for col in predictions_df.columns if col != 'Actual']

    # Check if the Excel file already exists
    if os.path.exists(file_path):
        wb = load_workbook(_ensure_excel_file(file_path))
        ws = wb.create_sheet(title="New Evaluation Features")
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = "evaluation_features_print"

    image_row = 1  # Where to start images

    for feature_to_plot in feature_names:
        feature_values = X_test[feature_to_plot]
        plot_df = predictions_df.copy()
        plot_df[feature_to_plot] = feature_values.values

        # Sort by current feature to plot lines well
        plot_df = plot_df.sort_values(by=feature_to_plot)

        # Begin Matplotlib plot
        plt.figure(figsize=(6, 4))
        # Plot each quantile as a line
        for quantile in quantile_columns:
            plt.plot(plot_df[feature_to_plot], plot_df[quantile], label=f"{quantile}")

        # Plot actual values as scatter
        plt.scatter(plot_df[feature_to_plot], plot_df['Actual'], color='black', s=12, label=f"Actual {TARGET_LABEL}")

        plt.title(f"Predicted Quantiles vs. {feature_to_plot}")
        plt.xlabel(feature_to_plot)
        plt.ylabel(f"{TARGET_LABEL}")
        plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
        plt.tight_layout()

        # Save figure to a BytesIO stream
        img_bytes = BytesIO()
        img_bytes
        plt.savefig(img_bytes, format="png")
        plt.close()
        img_bytes.seek(0)

        img = Image(img_bytes)
        img.anchor = f"A{image_row}"
        ws.add_image(img)

        image_row += 20  # Adjust as you like

    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

In [15]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def evaluate_uncertainity_matrix(predictions_df, quantiles, excel_file_path, model_name='Model'):
    predictions_df = rearrange_quantiles(predictions_df, excel_file_path)  # FIX Q2

   # Extract the actual target values
    y_true = predictions_df['Actual']

    # Open existing Excel file
    wb = load_workbook(_ensure_excel_file(excel_file_path))

    # Create unique sheet names
    def get_unique_sheet_name(wb, base_name):
        sheet_name = base_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base_name}_{i}"
            i += 1
        return sheet_name

    # Create separate sheets for plots and values
    plots_sheet_title = get_unique_sheet_name(wb, 'Plots')
    values_sheet_title = get_unique_sheet_name(wb, 'Values')

    ws_plots = wb.create_sheet(title=plots_sheet_title)
    ws_values = wb.create_sheet(title=values_sheet_title)
    # FIX Q2: how often the raw quantile predictions crossed (before sorting)
    _cr = quantile_crossing_report.get(excel_file_path, {})
    ws_values.append(['Rows with quantile crossing (before sorting)', _cr.get('rows_crossing'), 'of', _cr.get('rows'), 'rows', f"{_cr.get('pct_crossing')}%"])
    print(f"Quantile crossing before sorting: {_cr.get('rows_crossing')} of {_cr.get('rows')} rows ({_cr.get('pct_crossing')}%)")

    # Positions to place images
    image_positions = ['A1', 'A25', 'A49', 'A73']  # Adjust as needed

    ### 1. Validity / Coverage ###

    # Compute empirical coverage for each quantile
    empirical_quantiles = []
    for q in quantiles:
        coverage = (predictions_df[q] >= y_true).mean()
        empirical_quantiles.append(coverage)

    # Plot theoretical quantiles vs empirical quantiles
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=quantiles, y=quantiles, color="magenta", linestyle='--', linewidth=2, label="Ideal")
    sns.lineplot(x=quantiles, y=empirical_quantiles, color="blue", linewidth=2, label="Empirical")
    sns.scatterplot(x=quantiles, y=empirical_quantiles, color="blue", s=50)
    plt.xlabel("Nominal Quantile Levels")
    plt.ylabel("Empirical Quantile Levels")
    plt.title(f"Validity / Coverage - {model_name}")
    plt.legend()

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add the image to the plots worksheet
    ws_plots.add_image(img, image_positions[0])

    ### 2. Sharpness / Interval Length ###

    # Define coverage levels to evaluate
    coverage_levels = [0.6, 0.8, 0.9]
    average_interval_lengths = []

    # Compute average interval lengths for each coverage level
    for c in coverage_levels:
        lower_q = (1 - c) / 2
        upper_q = 1 - lower_q
        # Find the nearest quantiles in our quantiles list
        lower_quantile = min(quantiles, key=lambda x: abs(x - lower_q))
        upper_quantile = min(quantiles, key=lambda x: abs(x - upper_q))

        # Compute interval length for each observation
        interval_length = predictions_df[upper_quantile] - predictions_df[lower_quantile]
        avg_interval_length = interval_length.mean()
        average_interval_lengths.append(avg_interval_length)
        # Write interval length information to the values worksheet
        ws_values.append([f'Coverage level: {c*100:.0f}%', f'Interval: [{lower_quantile}, {upper_quantile}]', f'Average Interval Length: {avg_interval_length:.4f}'])

    # Plot coverage levels vs average interval lengths
    plt.figure(figsize=(10, 6))
    plt.plot([c*100 for c in coverage_levels], average_interval_lengths, marker='o')
    plt.xlabel('Coverage Level (%)')
    plt.ylabel('Average Interval Length')
    plt.title(f'Sharpness / Interval Length - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[1])

    ### 3. Negative Log-Likelihood (NLL) ###

    # Use the 5th and 95th percentiles to estimate standard deviation
    if 0.05 in quantiles and 0.95 in quantiles:
        z_lower = norm.ppf(0.05)  # z-score for 5% quantile (~ -1.6449)
        z_upper = norm.ppf(0.95)  # z-score for 95% quantile (~ +1.6449)

        # Extract quantile predictions
        q_lower = predictions_df[0.05]
        q_upper = predictions_df[0.95]

    elif 0.1 in quantiles and 0.9 in quantiles:
        z_lower = norm.ppf(0.1)  # z-score for 10% quantile (~ -1.2816)
        z_upper = norm.ppf(0.9)  # z-score for 90% quantile (~ +1.2816)

        # Extract quantile predictions
        q_lower = predictions_df[0.1]
        q_upper = predictions_df[0.9]
    else:
        raise ValueError("Required quantiles for NLL estimation not found in quantiles list.")

    # Use median as the mean estimate
    if 0.5 in quantiles:
        q_median = predictions_df[0.5]
    else:
        # If 0.5 quantile is not available, use the middle quantile
        q_median = predictions_df[quantiles[len(quantiles)/2]]

    # Estimate standard deviation for each observation
    std_estimates = (q_upper - q_lower) / (z_upper - z_lower)

    # Ensure standard deviations are positive and non-zero
    std_estimates = std_estimates.clip(lower=1e-6)

    # Extract mean estimates (median predictions)
    mean_estimates = q_median

    # Compute Negative Log-Likelihood for each observation
    nll = -norm.logpdf(y_true, loc=mean_estimates, scale=std_estimates)

    # Compute average NLL
    average_nll = nll.mean()
    # Write average NLL to the values worksheet
    ws_values.append(['Average Negative Log-Likelihood (NLL)', average_nll])

    # Write NLL values to the values worksheet
    ws_values.append(['NLL Values'])
    for value in nll:
        ws_values.append([value])

    # Plot ECDF of NLL values
    plt.figure(figsize=(10, 6))
    sns.ecdfplot(nll, color='blue', linewidth=2)
    plt.xlabel('Negative Log-Likelihood (NLL)')
    plt.ylabel('ECDF')
    plt.title(f'NLL ECDF - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[2])

    ### 4. Continuous Ranked Probability Score (CRPS) ###

    # Prepare ensemble of quantile predictions for each observation
    ensemble_predictions = predictions_df[quantiles].to_numpy()

    # Ensure quantile predictions are sorted for each observation
    ensemble_predictions.sort(axis=1)

    # Compute CRPS for each observation using ensemble predictions
    # FIX Q4: CRPS from a set of quantiles = 2 * integral of the pinball loss over tau.
    # Each quantile level gets a weight equal to the share of [0, 1] it represents
    # (levels are unevenly spaced). Was: ps.crps_ensemble, which treats the 13 quantiles as
    # 13 equally likely samples.
    _taus = np.array(quantiles, dtype=float)
    _edges = np.concatenate([[0.0], (_taus[1:] + _taus[:-1]) / 2.0, [1.0]])
    _w = np.diff(_edges)
    _err = np.asarray(y_true, dtype=float)[:, None] - ensemble_predictions
    _pinball = np.maximum(_taus * _err, (_taus - 1.0) * _err)
    crps_values = 2.0 * (_pinball * _w).sum(axis=1)

    # Compute average CRPS
    average_crps = crps_values.mean()
    # Write average CRPS to the values worksheet
    ws_values.append(['Average Continuous Ranked Probability Score (CRPS)', average_crps])

    # Write CRPS values to the values worksheet
    ws_values.append(['CRPS Values'])
    for value in crps_values:
        ws_values.append([value])

    # Plot ECDF of CRPS values
    plt.figure(figsize=(10, 6))
    sns.ecdfplot(crps_values, color='green', linewidth=2)
    plt.xlabel('Continuous Ranked Probability Score (CRPS)')
    plt.ylabel('ECDF')
    plt.title(f'CRPS ECDF - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[3])

    # Save the workbook
    _ensure_parent_dir(excel_file_path)
    wb.save(_ensure_parent_dir(excel_file_path))

    # Return a dictionary of results
    results = {
        'empirical_quantiles': empirical_quantiles,
        'average_interval_lengths': average_interval_lengths,
        'coverage_levels': coverage_levels,
        'average_nll': average_nll,
        'average_crps': average_crps,
        'nll_values': nll,
        'crps_values': crps_values
    }

    return results

In [16]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def calibrate_and_plot_intervals(predictions_df, excel_file_path, alpha=0.1, max_samples=1000, random_seed=42):
    predictions_df = rearrange_quantiles(predictions_df, excel_file_path)  # FIX Q2
    # Extract necessary columns
    lower = predictions_df[0.05].fillna(0)
    upper = predictions_df[0.95].fillna(0)
    pred = predictions_df[0.5].fillna(0)
    labels = predictions_df['Actual'].fillna(0)

    # Problem setup
    total_samples = labels.shape[0]
    n = min(max_samples, total_samples)  # Ensure n does not exceed the number of samples

    # Ensure there are enough samples for both calibration and validation
    if total_samples <= n:
        n = total_samples // 2

    # Split the data into calibration and validation sets
    np.random.seed(random_seed)  # Set the random seed for reproducibility
    idx = np.array([1] * n + [0] * (total_samples - n)) > 0
    np.random.shuffle(idx)
    cal_labels, val_labels = labels[idx], labels[~idx]
    cal_upper, val_upper = upper[idx], upper[~idx]
    cal_lower, val_lower = lower[idx], lower[~idx]
    cal_pred, val_pred = pred[idx], pred[~idx]

    # Calculate uncertainty intervals
    cal_U = cal_upper - cal_lower
    val_U = val_upper - val_lower

    # Avoid division by zero by replacing zero intervals with a small number
    cal_U[cal_U == 0] = np.finfo(float).eps
    val_U[val_U == 0] = np.finfo(float).eps

    # Get scores
    cal_scores = np.abs(cal_pred - cal_labels) / cal_U

    # Debugging: Check for NaN in scores
    if np.isnan(cal_scores).any():
        print("NaN values found in cal_scores. Check data integrity.")

    # Get the score quantile
    qhat = np.quantile(cal_scores, np.ceil((n + 1) * (1 - alpha)) / n, interpolation='higher')

    # Deploy (output=lower and upper adjusted quantiles)
    prediction_sets = [val_pred - val_U * qhat, val_pred + val_U * qhat]

    # Calculate empirical coverage (before and after calibration)
    prediction_sets_uncalibrated = [val_lower, val_upper]
    empirical_coverage_uncalibrated = ((val_labels >= prediction_sets_uncalibrated[0]) & (val_labels <= prediction_sets_uncalibrated[1])).mean() * 100
    print(f"The empirical coverage before calibration is: {empirical_coverage_uncalibrated:.2f}%")

    empirical_coverage = ((val_labels >= prediction_sets[0]) & (val_labels <= prediction_sets[1])).mean() * 100
    print(f"The empirical coverage after calibration is: {empirical_coverage:.2f}%")

    # Create a DataFrame with all the necessary data
    data = pd.DataFrame({
        'Index': np.arange(len(val_labels)),
        'Uncalibrated Lower': val_lower,
        'Uncalibrated Upper': val_upper,
        'Calibrated Lower': prediction_sets[0],
        'Calibrated Upper': prediction_sets[1],
        'True Label': val_labels,
        'Predicted Value': val_pred
    })

    plt.figure(figsize=(14, 8))

    # Plot the uncalibrated prediction intervals as a shaded area
    plt.fill_between(
        data['Index'],
        data['Uncalibrated Lower'],
        data['Uncalibrated Upper'],
        color='blue',
        alpha=0.5,
        label='Uncalibrated Prediction Interval'
    )

    # Plot the calibrated prediction intervals as a shaded area
    plt.fill_between(
        data['Index'],
        data['Calibrated Lower'],
        data['Calibrated Upper'],
        color='lightgreen',
        alpha=0.5,
        label='Calibrated Prediction Interval'
    )

    # Plot the true labels as points and connect them with a line
    plt.plot(
        data['Index'],
        data['True Label'],
        'o-',
        color='red',
        markersize=4,
        label='Actual Value'
    )

    # Plot the predicted values as points and connect them with a line
    plt.plot(
        data['Index'],
        data['Predicted Value'],
        'o-',
        color='black',
        markersize=4,
        label='Predicted Value'
    )

    plt.xlabel('Sample Number')
    plt.ylabel(f'{TARGET_LABEL}')
    plt.title('Calibrated and Uncalibrated Prediction Intervals with True and Predicted Values')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()

    # Save the plot as an image
    image_path = 'plot.png'
    _ensure_parent_dir(image_path)
    plt.savefig(_ensure_parent_dir(image_path))
    plt.close()

    # Load the existing Excel file
    workbook = load_workbook(_ensure_excel_file(excel_file_path))

    # Create a new sheet for the plot
    new_sheet_name = "calibrate_and_plot_intervals"
    workbook.create_sheet(title=new_sheet_name)

    # Insert the image into the new sheet
    sheet = workbook[new_sheet_name]
    img = Image(image_path)
    sheet.add_image(img, 'A1')

    # Save the workbook
    _ensure_parent_dir(excel_file_path)
    workbook.save(_ensure_parent_dir(excel_file_path))

In [17]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def plot_quantile_intervals_to_excel(predictions_df, file_path, alpha=0.1):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    """
    Plots conformal prediction intervals and their coverage, and saves the plots to an Excel file.

    Parameters:
    - predictions_df: DataFrame containing prediction quantiles and actual values.
    - file_path: Path to the Excel file where plots will be saved.
    - alpha: Desired coverage level (default is 0.1).
    """
    # Extract mean prediction and uncertainty
    mean_prediction = predictions_df[0.5]
    uncertainty = predictions_df[0.95] - predictions_df[0.05]
    actual_values = predictions_df['Actual']

    # Calculate conformal scores
    conformal_scores = np.abs(mean_prediction - actual_values) / uncertainty

    # Function to find weighted quantile
    def weighted_quantile(values, quantile, sample_weight=None):
        values = np.array(values)
        if sample_weight is None:
            sample_weight = np.ones(len(values))
        sorter = np.argsort(values)
        values, sample_weight = values[sorter], sample_weight[sorter]
        weighted_quantiles = np.cumsum(sample_weight) - 0.5 * sample_weight
        weighted_quantiles /= np.sum(sample_weight)
        return np.interp(quantile, weighted_quantiles, values)

    # Calculate weighted quantile
    weights = np.ones_like(conformal_scores)  # Uniform weights for simplicity
    quantile = weighted_quantile(conformal_scores, 1 - alpha, sample_weight=weights)

    # Calculate prediction intervals
    lower_bound = mean_prediction - quantile * uncertainty
    upper_bound = mean_prediction + quantile * uncertainty

    # Naive conformal prediction
    naive_quantile = np.quantile(conformal_scores, 1 - alpha)
    naive_lower_bound = mean_prediction - naive_quantile * uncertainty
    naive_upper_bound = mean_prediction + naive_quantile * uncertainty

    # Coverage calculation (not plotted, just for potential diagnostics)
    weighted_coverage_points = (actual_values >= lower_bound) & (actual_values <= upper_bound)
    naive_coverage_points = (actual_values >= naive_lower_bound) & (actual_values <= naive_upper_bound)

    # Visualization using Matplotlib
    fig, ax = plt.subplots(figsize=(10, 5))

    x = np.arange(len(predictions_df))
    # Actual Values
    ax.plot(x, actual_values, label='Actual', color='black', linewidth=2)

    # Mean Prediction
    ax.plot(x, mean_prediction, label='Mean Prediction', color='darkorange', linewidth=2)

    # Weighted Prediction Interval (blue fill)
    ax.fill_between(x, lower_bound, upper_bound, color='blue', alpha=0.3, label='Weighted Interval')
    # Optionally, plot the bounds as lines if you want:
    # ax.plot(x, lower_bound, color='blue', linewidth=1, linestyle='--')
    # ax.plot(x, upper_bound, color='blue', linewidth=1, linestyle='--')

    # Naive Prediction Interval (red fill)
    ax.fill_between(x, naive_lower_bound, naive_upper_bound, color='red', alpha=0.3, label='Naive Interval')
    # Optionally, plot the bounds as lines:
    # ax.plot(x, naive_lower_bound, color='red', linewidth=1, linestyle='--')
    # ax.plot(x, naive_upper_bound, color='red', linewidth=1, linestyle='--')

    ax.set_title('Quantile Prediction Intervals')
    ax.set_xlabel('Sample Number')
    ax.set_ylabel(f'{TARGET_LABEL}')
    ax.grid(False)
    ax.set_facecolor('white')

    # Place the legend outside the plot
    ax.legend(loc='center left', bbox_to_anchor=(1.05, 0.5), borderaxespad=0.0)

    plt.tight_layout(rect=[0,0,0.80,1.0])  # Make space for legend

    # Save the plot to a BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Add image to Excel
    if os.path.exists(file_path):
        wb = load_workbook(_ensure_excel_file(file_path))
        ws = wb.create_sheet(title="plot_quantile_intervals_to_excel")
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = "plot_quantile_intervals_to_excel"

    img = Image(img_data)
    img.anchor = 'A1'
    ws.add_image(img)
    ws.column_dimensions['A'].width = 30
    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

# **Tuned Parameters**

In [18]:
# FIX K2: load the tuning results saved by hyperparameter_tuning/Optuna_autosampler.ipynb
# (was: a hand-pasted copy of an older tuning run that had been selected on the TEST set).
import pickle
if IN_KAGGLE:
    _bp = glob.glob("/kaggle/input/**/hyperparameter_tuning/best_params.pkl", recursive=True)[0]
else:
    _bp = "./drive/MyDrive/Regression/streamflow_uncertainty_analysis/hyperparameter_tuning/best_params.pkl"
with open(_bp, "rb") as f:
    best_scores_autosampler = pickle.load(f)
print("Loaded tuning results from", _bp, "-", len(best_scores_autosampler), "model x pruner entries")


Loaded tuning results from /kaggle/input/datasets/daneshjangra/streamflow-tuned-params/hyperparameter_tuning/best_params.pkl - 80 model x pruner entries


In [19]:
best_scores_autosampler

{('Random Forest', 'MedianPruner'): {'best_score': 13226.396272512651,
  'cv_rmse': 115.00607058982865,
  'best_params': {'n_estimators': 200,
   'criterion': 'absolute_error',
   'max_depth': 20,
   'min_samples_split': 0.01,
   'min_samples_leaf': 3,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 1.0,
   'max_leaf_nodes': 50,
   'min_impurity_decrease': 0.1,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   'ccp_alpha': 0.1},
  'test_mse': 12859.83788971354,
  'test_rmse': np.float64(113.40122525666793),
  'test_corr_coef': np.float64(0.8221718688883797),
  'pruner': 'MedianPruner'},
 ('Random Forest', 'NopPruner'): {'best_score': 11546.835875160255,
  'cv_rmse': 107.45620445167536,
  'best_params': {'n_estimators': 200,
   'criterion': 'absolute_error',
   'max_depth': 20,
   'min_samples_split': 0.01,
   'min_samples_leaf': 1,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 1.0,
   'max_leaf_nodes': 50,
   'min_impurity_decrease': 0.

# **Quantile Regression with Lightgbm**

In [20]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_LGBM_df = pd.DataFrame()


# Train a model for each quantile and make predictions using LightGBM
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Update params with quantile-specific settings
    params.update({
        'objective': 'quantile',
        'alpha': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = LGBMRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_LGBM_df[q] = predictions

# Add actual target values to the DataFrame
predictions_LGBM_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_LGBM_df.head())

         0.05         0.1        0.15         0.2         0.3         0.4  \
0  125.121689  153.146457  149.251932  138.698492  151.054428  141.091592   
1  161.561412  175.817045  184.911927  201.774315  208.777244  208.599359   
2  132.848206  121.976927  131.695037  149.817794  155.517449  180.409774   
3  116.794630  129.043321  124.229127  117.030123  135.494687  141.802089   
4  122.447159  123.631348  125.342757  117.082837  115.827972  133.762617   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0  158.992818  154.588134  174.877749  188.665725  182.759956  282.045321   
1  213.525073  252.191632  245.640572  318.248929  305.264582  329.205416   
2  199.071304  183.090970  184.491408  225.118589  262.327327  283.772406   
3  148.700111  140.361675  166.857487  164.081093  175.859479  171.945968   
4  155.112161  181.553403  178.399656  176.090558  191.498159  216.394240   

         0.95  Actual  
0  348.476103     160  
1  348.382590     190  
2 

In [21]:
quantile_regression_print(predictions_LGBM_df,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

First few actual values: [160 190 180 110 200]
Coverage of 90% prediction interval: 73.96%


In [22]:
evaluation_features_print(predictions_LGBM_df,x_test,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

In [23]:
evaluate_uncertainity_matrix(predictions_LGBM_df, quantiles, "./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/LightGBM.xlsx", model_name='LightGBM')

Quantile crossing before sorting: 96 of 96 rows (100.0%)


{'empirical_quantiles': [np.float64(0.11458333333333333),
  np.float64(0.19791666666666666),
  np.float64(0.2604166666666667),
  np.float64(0.3333333333333333),
  np.float64(0.40625),
  np.float64(0.4583333333333333),
  np.float64(0.4791666666666667),
  np.float64(0.5625),
  np.float64(0.6354166666666666),
  np.float64(0.6875),
  np.float64(0.7395833333333334),
  np.float64(0.8125),
  np.float64(0.8541666666666666)],
 'average_interval_lengths': [np.float64(84.07194284887036),
  np.float64(140.13303676031668),
  np.float64(180.31987753474445)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(6.2028491590974495),
 'average_crps': np.float64(52.16844252822492),
 'nll_values': array([ 5.14007656,  5.04409322,  4.8164479 ,  4.89933189,  5.22310675,
         9.03998717,  6.28738773,  4.74340471,  5.58409406,  5.33514433,
         4.7652136 ,  6.58942087,  6.30534756,  6.62470402,  6.42479296,
         9.76548922,  6.55124624,  5.18385195,  6.00117425,  5.03074154,
         7

In [24]:
calibrate_and_plot_intervals(predictions_LGBM_df, "./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

The empirical coverage before calibration is: 79.17%
The empirical coverage after calibration is: 91.67%


In [25]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_LGBM_df,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

# **Quantile Regression with XGBoost**

In [26]:
# Get the best hyperparameters for XGBRegressor
best_paramst = get_best_model_params(best_scores_autosampler, 'XGBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_XGB_df = pd.DataFrame()

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_paramst.copy()  # FIX Q1: was best_params (= LightGBM's settings from the cell above)

    # Adjust according to your XGBoost version
    params.update({
        'objective': 'reg:quantileerror',  # Use the correct objective
        'quantile_alpha': q,                         # Use 'alpha' to set the quantile level
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = XGBRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_XGB_df[q] = predictions

# Add actual target values to the DataFrame
predictions_XGB_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_XGB_df.head())

         0.05         0.1        0.15         0.2         0.3         0.4  \
0  121.319138  135.398392  142.269989  135.642548  137.472900  125.340233   
1  135.653427  160.952286  167.000977  182.594620  196.514160  202.022202   
2  102.147621  117.240143  126.000221  143.989502  169.555984  170.436066   
3  108.068748   98.501991  100.360283  104.449013  132.425049  151.797150   
4  108.582680  115.170906  119.701653  117.608841  121.245171  126.247765   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0  133.553497  130.887756  139.506790  153.899933  194.987686  231.313889   
1  217.383484  268.799713  272.227203  273.497528  320.865753  320.743774   
2  165.656448  190.769287  194.452881  192.574524  219.983673  288.509979   
3  157.346024  167.052856  169.962677  186.396225  187.556030  208.052460   
4  134.675659  141.857391  142.938354  177.027084  218.978912  259.903595   

         0.95  Actual  
0  375.011963     160  
1  353.045349     190  
2 

In [27]:
quantile_regression_print(predictions_XGB_df,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

First few actual values: [160 190 180 110 200]
Coverage of 90% prediction interval: 83.33%


In [28]:
evaluation_features_print(predictions_XGB_df,x_test,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

In [29]:
evaluate_uncertainity_matrix(predictions_XGB_df, quantiles, "./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/XGBoost.xlsx", model_name='XGBoost')

Quantile crossing before sorting: 87 of 96 rows (90.62%)


{'empirical_quantiles': [np.float64(0.08333333333333333),
  np.float64(0.13541666666666666),
  np.float64(0.20833333333333334),
  np.float64(0.25),
  np.float64(0.3333333333333333),
  np.float64(0.4270833333333333),
  np.float64(0.46875),
  np.float64(0.5625),
  np.float64(0.6145833333333334),
  np.float64(0.6770833333333334),
  np.float64(0.75),
  np.float64(0.8333333333333334),
  np.float64(0.9166666666666666)],
 'average_interval_lengths': [np.float64(85.60442479451497),
  np.float64(172.7503391901652),
  np.float64(243.53349709510803)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(6.029057965029992),
 'average_crps': np.float64(53.18432455162207),
 'nll_values': array([ 5.30692982,  5.19569865,  4.97008376,  5.43522712,  5.59840059,
         8.01956977,  6.63971807,  4.83660598,  5.76600225,  5.45283048,
         5.09559208,  5.53398523,  6.34415574,  5.33035084,  6.2039357 ,
         8.09317118,  5.78224983,  5.17227582,  5.77869815,  5.56254117,
         5.9231

In [30]:
calibrate_and_plot_intervals(predictions_XGB_df, "./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

The empirical coverage before calibration is: 87.50%
The empirical coverage after calibration is: 97.92%


In [31]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_XGB_df, "./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

# **Quantile Regression with GPBoost**

In [32]:
# Get the best hyperparameters for GPBoostRegressor
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_GPBoost_df = pd.DataFrame()

# Parameters that may conflict with quantile-specific settings
incompatible_keys = ['objective', 'alpha', 'quantile_alpha', 'boosting_type', 'metric', 'eval_metric']

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Remove incompatible parameters
    for key in incompatible_keys:
        params.pop(key, None)

    # Update params with quantile-specific settings
    params.update({
        'objective': 'quantile',
        'alpha': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = GPBoostRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_GPBoost_df[q] = predictions

# Add actual target values to the DataFrame
predictions_GPBoost_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_GPBoost_df.head())

         0.05         0.1        0.15         0.2         0.3         0.4  \
0   89.878960   99.117555  109.302993  116.545635  120.693485  135.231409   
1  128.320372  159.550335  168.467548  185.341480  193.920949  206.937810   
2   91.154853   99.728578  108.856333  125.067931  131.600848  144.394110   
3   89.571872  105.986737  114.859972  118.678687  136.217177  156.865479   
4  119.678810  118.691540  117.077637  117.974760  119.771232  128.886692   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0  144.807960  156.819974  171.787974  186.077456  200.393868  230.506288   
1  217.862342  240.474093  282.253263  308.911468  342.137638  362.017514   
2  159.512096  167.128939  176.419157  187.488791  215.974753  226.689016   
3  164.867416  170.624341  207.683802  244.724830  230.674854  288.711123   
4  136.815511  162.563124  190.492986  194.087289  216.413143  242.236604   

         0.95  Actual  
0  419.347094     160  
1  431.879646     190  
2 

In [33]:
quantile_regression_print(predictions_GPBoost_df,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

First few actual values: [160 190 180 110 200]
Coverage of 90% prediction interval: 91.67%


In [34]:
evaluation_features_print(predictions_GPBoost_df,x_test,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

In [35]:
evaluate_uncertainity_matrix(predictions_GPBoost_df, quantiles,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/GPBoost.xlsx", model_name='GPBoost')

Quantile crossing before sorting: 47 of 96 rows (48.96%)


{'empirical_quantiles': [np.float64(0.041666666666666664),
  np.float64(0.10416666666666667),
  np.float64(0.19791666666666666),
  np.float64(0.25),
  np.float64(0.3020833333333333),
  np.float64(0.4270833333333333),
  np.float64(0.4479166666666667),
  np.float64(0.5520833333333334),
  np.float64(0.6458333333333334),
  np.float64(0.7395833333333334),
  np.float64(0.84375),
  np.float64(0.8958333333333334),
  np.float64(0.9583333333333334)],
 'average_interval_lengths': [np.float64(106.992119815061),
  np.float64(183.36019412592248),
  np.float64(298.72171393420723)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(6.016696905373055),
 'average_crps': np.float64(50.27373249512849),
 'nll_values': array([ 5.53712468,  5.48930298,  5.31236853,  5.46202431,  5.59771132,
         6.78911838,  6.06676228,  5.15683086,  5.7834126 ,  5.74006285,
         5.38248373,  5.55188096,  5.37705334,  5.59515541,  6.00393567,
         7.28455478,  5.82493816,  5.29253856,  5.74460118,  

In [36]:
calibrate_and_plot_intervals(predictions_GPBoost_df,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

The empirical coverage before calibration is: 93.75%
The empirical coverage after calibration is: 97.92%


In [37]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_GPBoost_df,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

# **Quantile Regression with NGBoost**

In [38]:
# Correct best_params assignment
best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')

# DataFrame to store predictions
predictions_NGB_df = pd.DataFrame()

# Remove 'Score' and 'Dist' if present in best_params
for key in ['Score', 'Dist']:
    best_params.pop(key, None)

# Update params with model settings
best_params.update({
    'Dist': Normal,
    'Score': LogScore,
    'random_state': 42,
    'verbose': True
})

# Initialize and fit the NGBoost model once
model = NGBRegressor(**best_params)

# Fit the model
model.fit(X_train, y_train)

# Make predictions
pred_dist = model.pred_dist(X_test)

# Extract the mean (mu) and standard deviation (sigma) of the predicted distributions
mu = pred_dist.loc    # Mean predictions
sigma = pred_dist.scale  # Standard deviation predictions

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# For each quantile, compute predictions
for q in quantiles:
    # Calculate the z-score for the desired quantile
    z = norm.ppf(q)

    # Compute the quantile prediction per sample
    predictions = mu + z * sigma

    # Add predictions to the DataFrame
    predictions_NGB_df[q] = predictions

# Add actual target values to the DataFrame
predictions_NGB_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_NGB_df.head())

[iter 0] loss=6.9234 val_loss=0.0000 scale=2.0000 norm=273.4345


[iter 100] loss=5.9647 val_loss=0.0000 scale=2.0000 norm=115.8059


         0.05         0.1        0.15         0.2         0.3         0.4  \
0   55.590633   81.391042   98.798455  112.633315  135.161193  154.410433   
1  140.529778  166.356234  183.781221  197.630048  220.180668  239.449342   
2   67.490028   92.987545  110.190600  123.863042  146.126447  165.149706   
3   42.022874   66.115327   82.370390   95.289398  116.325957  134.300920   
4   34.095485   59.440944   76.541404   90.132308  112.262942  131.172752   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0  172.402235  190.394038  209.643278  232.171156  246.006016  263.413429   
1  257.459308  275.469273  294.737947  317.288567  331.137395  348.562382   
2  182.930288  200.710871  219.734130  241.997535  255.669977  272.873031   
3  151.101686  167.902453  185.877416  206.913974  219.832983  236.088046   
4  148.847296  166.521841  185.431651  207.562284  221.153188  238.253649   

         0.95  Actual  
0  289.213838     160  
1  374.388837     190  
2 

In [39]:
quantile_regression_print(predictions_NGB_df,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

First few actual values: [160 190 180 110 200]
Coverage of 90% prediction interval: 91.67%


In [40]:
evaluation_features_print(predictions_NGB_df,x_test,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

In [41]:
evaluate_uncertainity_matrix(predictions_NGB_df, quantiles,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/NGBoost.xlsx", model_name='NGBoost')

Quantile crossing before sorting: 0 of 96 rows (0.0%)


{'empirical_quantiles': [np.float64(0.020833333333333332),
  np.float64(0.08333333333333333),
  np.float64(0.10416666666666667),
  np.float64(0.16666666666666666),
  np.float64(0.3125),
  np.float64(0.4375),
  np.float64(0.5416666666666666),
  np.float64(0.6145833333333334),
  np.float64(0.71875),
  np.float64(0.8333333333333334),
  np.float64(0.8645833333333334),
  np.float64(0.8958333333333334),
  np.float64(0.9375)],
 'average_interval_lengths': [np.float64(130.49169940821108),
  np.float64(198.70202294828405),
  np.float64(255.03128544826802)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(6.123513307226276),
 'average_crps': np.float64(52.14111123771996),
 'nll_values': array([ 5.19709893,  5.63311495,  5.17091193,  5.30542933,  5.43286702,
         6.68541284,  6.57904435,  5.24208437,  6.04307729,  5.75195513,
         5.35003791,  5.57569502,  5.39518963,  5.23780432,  5.95288437,
         6.73286834,  6.35139979,  5.3927786 ,  5.78104819,  5.25845432,
       

In [42]:
calibrate_and_plot_intervals(predictions_NGB_df,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

The empirical coverage before calibration is: 91.67%
The empirical coverage after calibration is: 97.92%


In [43]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_NGB_df,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

# **Quantile Regression with Gradient Boosting**

In [44]:
best_params = get_best_model_params(best_scores_autosampler, 'GradientBoosting')


# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_GBR_df = pd.DataFrame()

# Custom quantile loss function
def quantile_loss(q, y_true, y_pred):
    e = y_true - y_pred
    return np.mean(np.maximum(q * e, (q - 1) * e))

# Train a model for each quantile and make predictions using GradientBoostingRegressor
for q in quantiles:

    params = best_params.copy()

    params.update({
        'loss': 'quantile',
        'alpha': q,
        'random_state': 42
    })
    # Initialize the model with quantile-specific settings
    model = GradientBoostingRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_GBR_df[q] = predictions

# Add actual target values to the DataFrame
predictions_GBR_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_GBR_df.head())

        0.05        0.1        0.15      0.2         0.3         0.4  \
0  86.456363  109.99999  109.913827  116.080  137.692618  138.400671   
1  86.456363  109.99999  150.130467  172.576  190.002749  208.801311   
2  86.456363  109.99999  109.913827  116.080  150.594698  153.814667   
3  86.456363  109.99999  109.913827  116.080  120.918842  119.540101   
4  86.456363  109.99999  125.522467  125.680  118.743055  119.540101   

          0.5         0.6         0.7        0.8        0.85         0.9  \
0  145.960570  154.571482  178.933874  190.93843  226.225846  264.744875   
1  213.493978  227.867850  236.313074  239.33843  274.225846  296.744875   
2  161.573053  161.371482  189.333874  211.33843  226.225846  264.744875   
3  139.596532  143.285082  168.694386  190.93843  226.225846  264.744875   
4  127.291764  151.285082  168.694386  190.93843  226.225846  264.744875   

         0.95  Actual  
0  470.866158     160  
1  470.866158     190  
2  470.866158     180  
3  470.866158 

In [45]:
quantile_regression_print(predictions_GBR_df,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

First few actual values: [160 190 180 110 200]
Coverage of 90% prediction interval: 94.79%


In [46]:
evaluation_features_print(predictions_GBR_df,x_test,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

In [47]:
evaluate_uncertainity_matrix(predictions_GBR_df, quantiles,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx", model_name='GBM')

Quantile crossing before sorting: 79 of 96 rows (82.29%)


{'empirical_quantiles': [np.float64(0.020833333333333332),
  np.float64(0.052083333333333336),
  np.float64(0.11458333333333333),
  np.float64(0.15625),
  np.float64(0.3645833333333333),
  np.float64(0.3854166666666667),
  np.float64(0.4270833333333333),
  np.float64(0.4895833333333333),
  np.float64(0.5729166666666666),
  np.float64(0.7083333333333334),
  np.float64(0.8125),
  np.float64(0.8645833333333334),
  np.float64(0.96875)],
 'average_interval_lengths': [np.float64(111.12575142856048),
  np.float64(207.03367769407086),
  np.float64(406.5747165578073)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(6.060685275518156),
 'average_crps': np.float64(54.7893600351019),
 'nll_values': array([ 5.68706674,  5.70006108,  5.69255638,  5.71192489,  5.87343016,
         6.61923789,  6.26642691,  5.72326774,  5.87763674,  5.70892061,
         5.69686283,  5.77813836,  5.79845619,  5.71202701,  6.05645413,
         6.85292306,  5.89504037,  5.70006108,  5.88057765,  5.726303

In [48]:
calibrate_and_plot_intervals(predictions_GBR_df,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

The empirical coverage before calibration is: 95.83%
The empirical coverage after calibration is: 97.92%


In [49]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_GBR_df,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

# **Quantile Regression with CatBoost**

In [50]:
# Get the best hyperparameters for CatBoostRegressor
best_params = get_best_model_params(best_scores_autosampler, 'CatBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_CatBoost_df = pd.DataFrame()

# Parameters that may conflict with quantile-specific settings
incompatible_keys = ['objective', 'loss_function', 'eval_metric']

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Remove incompatible parameters
    for key in incompatible_keys:
        params.pop(key, None)

    # Update params with quantile-specific settings
    params.update({
        'loss_function': 'Quantile:alpha={}'.format(q),
        'random_seed': 42
    })

    # Initialize the model with the updated parameters
    model = CatBoostRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train, verbose=0)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_CatBoost_df[q] = predictions

# Add actual target values to the DataFrame
predictions_CatBoost_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_CatBoost_df.head())

         0.05         0.1        0.15         0.2         0.3         0.4  \
0  114.268015  121.874550  129.201887  133.895707  141.455686  143.003239   
1  127.266185  152.384373  165.897717  181.703457  204.505297  213.029543   
2   95.480850  114.345835  121.925405  130.195804  131.161251  155.432435   
3  101.345807  105.832062  103.908214  113.445871  120.982707  122.248433   
4  105.240301  110.003668  128.190727  126.157985  121.666108  126.693522   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0  150.398025  162.050759  172.175666  194.947190  198.520029  209.658296   
1  226.290042  234.030755  260.745443  291.204487  314.510588  334.839861   
2  162.673779  175.780784  194.112513  195.144260  223.664968  234.085248   
3  133.558076  137.930163  159.358256  191.140131  203.165651  201.382919   
4  140.987162  147.327725  168.525077  192.854226  219.351659  236.813807   

         0.95  Actual  
0  369.455737     160  
1  343.212104     190  
2 

In [51]:
quantile_regression_print(predictions_CatBoost_df,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

First few actual values: [160 190 180 110 200]
Coverage of 90% prediction interval: 83.33%


In [52]:
evaluation_features_print(predictions_CatBoost_df,x_test,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

In [53]:
evaluate_uncertainity_matrix(predictions_CatBoost_df, quantiles,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/CatBoost.xlsx", model_name='CatBoost')

Quantile crossing before sorting: 50 of 96 rows (52.08%)


{'empirical_quantiles': [np.float64(0.08333333333333333),
  np.float64(0.13541666666666666),
  np.float64(0.14583333333333334),
  np.float64(0.22916666666666666),
  np.float64(0.3229166666666667),
  np.float64(0.3854166666666667),
  np.float64(0.4479166666666667),
  np.float64(0.5),
  np.float64(0.5833333333333334),
  np.float64(0.71875),
  np.float64(0.7916666666666666),
  np.float64(0.8645833333333334),
  np.float64(0.9166666666666666)],
 'average_interval_lengths': [np.float64(89.05792037096347),
  np.float64(149.0804049279913),
  np.float64(233.36711959675236)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(6.05016358887422),
 'average_crps': np.float64(48.759548436168096),
 'nll_values': array([ 5.27780041,  5.25598417,  5.23828597,  4.85460194,  5.65488964,
         7.62623496,  6.90367974,  5.20302842,  5.77574695,  5.61843165,
         5.15603623,  5.30680421,  5.52417912,  5.09715282,  6.07061612,
         7.86567744,  5.97483342,  5.1602915 ,  5.61743225,  5

In [54]:
calibrate_and_plot_intervals(predictions_CatBoost_df,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

The empirical coverage before calibration is: 89.58%
The empirical coverage after calibration is: 95.83%


In [55]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_CatBoost_df,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

# **Quantile Regression with HistGradientBoosting**

In [56]:

# Assume best_params is a dictionary containing the best parameters for HistGradientBoostingRegressor
best_params = get_best_model_params(best_scores_autosampler, 'HGBR')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_HGBR_df = pd.DataFrame()

# Train a model for each quantile and make predictions using HistGradientBoostingRegressor
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Update params with quantile-specific settings
    params.update({
        'loss': 'quantile',
        'quantile': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = HistGradientBoostingRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_HGBR_df[q] = predictions

# Add actual target values to the DataFrame
predictions_HGBR_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_HGBR_df.head())

         0.05         0.1        0.15         0.2         0.3         0.4  \
0   93.457972   97.767258  106.105686  122.505100  130.754542  140.054546   
1  111.393805  142.840220  166.347831  190.223238  204.860909  202.840426   
2   88.168754   96.905371  105.902299  126.504863  140.826070  140.814033   
3   86.097432   98.618441  113.619508  116.379772  130.584944  139.053643   
4  126.533073  131.412095  136.738832  124.218261  122.618943  133.288184   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0  145.981026  158.455047  165.579641  191.294104  207.596369  232.919352   
1  212.744076  222.946672  263.038387  304.522800  340.807727  359.839388   
2  158.137361  166.156424  178.506786  194.506038  218.755579  241.274757   
3  147.622397  171.645013  217.757063  233.164856  226.359172  238.256299   
4  136.953444  153.730607  162.244534  195.890021  211.629783  245.669368   

         0.95  Actual  
0  429.117437     160  
1  357.331399     190  
2 

In [57]:
quantile_regression_print(predictions_HGBR_df,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/HGBM.xlsx")

First few actual values: [160 190 180 110 200]
Coverage of 90% prediction interval: 92.71%


In [58]:
evaluation_features_print(predictions_HGBR_df,x_test,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/HGBM.xlsx")

In [59]:
evaluate_uncertainity_matrix(predictions_HGBR_df, quantiles,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/HGBM.xlsx", model_name='HGBM')

Quantile crossing before sorting: 55 of 96 rows (57.29%)


{'empirical_quantiles': [np.float64(0.010416666666666666),
  np.float64(0.08333333333333333),
  np.float64(0.14583333333333334),
  np.float64(0.21875),
  np.float64(0.3020833333333333),
  np.float64(0.3958333333333333),
  np.float64(0.4375),
  np.float64(0.5104166666666666),
  np.float64(0.6041666666666666),
  np.float64(0.7604166666666666),
  np.float64(0.84375),
  np.float64(0.8645833333333334),
  np.float64(0.9375)],
 'average_interval_lengths': [np.float64(119.19951031053222),
  np.float64(192.07007162765845),
  np.float64(323.7050090053172)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(6.032356598267988),
 'average_crps': np.float64(51.612400154847045),
 'nll_values': array([ 5.55367595,  5.28871184,  5.50600378,  5.36013816,  5.65821108,
         6.88799851,  5.49464343,  5.30876239,  5.69947817,  5.72962354,
         5.42630699,  5.67960568,  5.46956637,  5.61633177,  6.05347015,
         7.93310052,  5.74270492,  5.2981937 ,  5.83645292,  5.65760032,
       

In [60]:
calibrate_and_plot_intervals(predictions_HGBR_df, "./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/HGBM.xlsx")

The empirical coverage before calibration is: 93.75%
The empirical coverage after calibration is: 97.92%


In [61]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_HGBR_df, "./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/HGBM.xlsx")

# **Quantile Regression with TabNet**

In [62]:
best_params = get_best_model_params(best_scores_autosampler, 'TabNet')

# Quantiles
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Storage
models_quantile = {}
predictions_TabNet_df = pd.DataFrame()

# Convert data to numpy (TabNet requirement)
X_train_np = np.array(X_train)
y_train_np = np.array(y_train).reshape(-1, 1)
X_test_np  = np.array(X_test)

# Quantile loss (pinball loss)
def quantile_loss(q):
    def loss(y_pred, y_true):
        error = y_true - y_pred
        return torch.mean(torch.maximum(q * error, (q - 1) * error))
    return loss

# FIX Q3: early stopping on a 20% validation split of TRAIN, scored with the pinball loss of
# the quantile being trained (was: eval_set = the training data itself, scored with MSE,
# so early stopping never triggered).
from pytorch_tabnet.metrics import Metric
X_fit_np, X_val_np, y_fit_np, y_val_np = train_test_split(X_train_np, y_train_np, test_size=0.2, random_state=42)

def pinball_metric(q):
    class Pinball(Metric):
        def __init__(self):
            self._name = f'pinball_{q}'
            self._maximize = False
        def __call__(self, y_true, y_score, weights=None):  # FIX B8: pytorch-tabnet2 also passes sample weights
            e = np.asarray(y_true, dtype=float) - np.asarray(y_score, dtype=float)
            return float(np.mean(np.maximum(q * e, (q - 1) * e)))
    return Pinball

# Train one TabNet per quantile
for q in quantiles:
    model = TabNetRegressor(
        **best_params,
    )

    model.fit(
        X_fit_np,
        y_fit_np,
        eval_set=[(X_val_np, y_val_np)],
        eval_metric=[pinball_metric(q)],
        loss_fn=quantile_loss(q),
        max_epochs=200,
        patience=30,
        batch_size=1024,
        virtual_batch_size=128,
        num_workers=0,
        drop_last=False
    )

    # Store model
    models_quantile[q] = model

    # Predict
    preds = model.predict(X_test_np).ravel()
    predictions_TabNet_df[q] = preds

# Add actual values
predictions_TabNet_df['Actual'] = y_test.values.ravel()

print(predictions_TabNet_df.head())

Stop training because you reached max_epochs = 200 with best_epoch = 190 and best_val_0_pinball_0.05 = 5.3762



Early stopping occurred at epoch 164 with best_epoch = 134 and best_val_0_pinball_0.1 = 12.22007



Early stopping occurred at epoch 174 with best_epoch = 144 and best_val_0_pinball_0.15 = 14.83429



Early stopping occurred at epoch 165 with best_epoch = 135 and best_val_0_pinball_0.2 = 19.38923



Early stopping occurred at epoch 185 with best_epoch = 155 and best_val_0_pinball_0.3 = 21.58942



Early stopping occurred at epoch 175 with best_epoch = 145 and best_val_0_pinball_0.4 = 28.71765


... (3 outputs left out)


Early stopping occurred at epoch 144 with best_epoch = 114 and best_val_0_pinball_0.8 = 25.50041



Early stopping occurred at epoch 190 with best_epoch = 160 and best_val_0_pinball_0.85 = 17.06607



Early stopping occurred at epoch 198 with best_epoch = 168 and best_val_0_pinball_0.9 = 13.92754



Early stopping occurred at epoch 165 with best_epoch = 135 and best_val_0_pinball_0.95 = 6.9473
         0.05         0.1        0.15         0.2         0.3         0.4  \
0  132.071747  139.520416  138.981995  140.945038  124.809418  143.001892   
1  124.644638  154.752762  149.935562  165.787628  170.083466  163.797821   
2  131.411469   88.862503  139.703339  161.631393  166.941666  140.414764   
3   90.891479   95.627670   98.957451  112.589752  101.881531  141.575317   
4  107.306946  102.089752  101.233253  134.513321  119.566444  153.341095   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0  142.569824  145.614288  184.627228  166.848114  157.103668  194.534668   
1  229.667023  292.665588  234.813538  247.061859  232.367859  274.347626   
2  140.248047  149.854126  195.003815  190.141190  305.104523  250.858353   
3  133.139465  140.753143  133.970688  178.056824  157.295990  182.026581   
4  118.639206  137.454575  193.633575  162.329010  228.

In [63]:
quantile_regression_print(predictions_TabNet_df,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/TabNet.xlsx")

First few actual values: [160 190 180 110 200]
Coverage of 90% prediction interval: 86.46%


In [64]:
evaluation_features_print(predictions_TabNet_df,x_test,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/TabNet.xlsx")

In [65]:
evaluate_uncertainity_matrix(predictions_TabNet_df, quantiles, "./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/TabNet.xlsx", model_name='TabNet')

Quantile crossing before sorting: 96 of 96 rows (100.0%)


{'empirical_quantiles': [np.float64(0.03125),
  np.float64(0.08333333333333333),
  np.float64(0.14583333333333334),
  np.float64(0.25),
  np.float64(0.3333333333333333),
  np.float64(0.40625),
  np.float64(0.46875),
  np.float64(0.5416666666666666),
  np.float64(0.6458333333333334),
  np.float64(0.75),
  np.float64(0.8229166666666666),
  np.float64(0.8541666666666666),
  np.float64(0.8958333333333334)],
 'average_interval_lengths': [np.float64(100.85598723093669),
  np.float64(163.66843152046204),
  np.float64(218.39931281407675)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(6.103911132471907),
 'average_crps': np.float64(50.01183733661969),
 'nll_values': array([ 4.57123813,  5.18054183,  5.14358242,  4.8870158 ,  5.56294471,
        10.84684045,  6.34678701,  5.42557644,  5.38706082,  5.69651701,
         5.09460742,  6.23452029,  6.06262214,  4.81725805,  6.59292718,
        12.1832325 ,  6.15603447,  5.21552243,  5.79216572,  5.36150156,
         6.96626751,  5.

In [66]:
calibrate_and_plot_intervals(predictions_TabNet_df, "./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/TabNet.xlsx")

The empirical coverage before calibration is: 85.42%
The empirical coverage after calibration is: 87.50%


In [67]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_TabNet_df,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/TabNet.xlsx")

# **Quantile Regression with PGBM**

In [68]:
class PGBMWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, **params):
        self.params = params
        self.model = None

    def fit(self, X, y):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        y_ = y.to_numpy() if hasattr(y, "to_numpy") else np.array(y)
        self.model = PGBM()
        self.model.train(
            train_set=(X_, y_),
            objective=mseloss_objective,
            metric=rmseloss_metric,
            params=self.params
        )
        return self

    def predict(self, X):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        return self.model.predict(X_).numpy()


In [69]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'PGBM')

# Remove any conflicting parameters from best_params
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize PGBM model
pgbm_model = PGBM()
def mseloss_objective(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian

def rmseloss_metric(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

# Fit the model
pgbm_model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params=best_params)

# Predict the distribution
pred_dist = pgbm_model.predict_dist(X_test)

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# DataFrame to store predictions
predictions_PGBM_df = pd.DataFrame()

# Calculate and store quantiles
for q in quantiles:
    predictions_PGBM_df[q] = np.quantile(pred_dist, q, axis=0)

# Add actual target values to the DataFrame
predictions_PGBM_df['Actual'] = y_test.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_PGBM_df.head())

Training on CPU


Estimator 0/100, Train metric: 272.0730
Estimator 1/100, Train metric: 260.4306
Estimator 2/100, Train metric: 249.5487
Estimator 3/100, Train metric: 239.3290
Estimator 4/100, Train metric: 229.7705
Estimator 5/100, Train metric: 220.8012
Estimator 6/100, Train metric: 212.4063
Estimator 7/100, Train metric: 204.5448
Estimator 8/100, Train metric: 197.0444


Estimator 9/100, Train metric: 190.0085
Estimator 10/100, Train metric: 183.3679
Estimator 11/100, Train metric: 177.1315
Estimator 12/100, Train metric: 171.2754
Estimator 13/100, Train metric: 165.7105
Estimator 14/100, Train metric: 160.5106
Estimator 15/100, Train metric: 155.6174
Estimator 16/100, Train metric: 150.9492
Estimator 17/100, Train metric: 146.5276
Estimator 18/100, Train metric: 142.3985
Estimator 19/100, Train metric: 138.4901
Estimator 20/100, Train metric: 134.7193
Estimator 21/100, Train metric: 131.1131
Estimator 22/100, Train metric: 127.7327


Estimator 23/100, Train metric: 124.5207
Estimator 24/100, Train metric: 121.4783
Estimator 25/100, Train metric: 118.5194
Estimator 26/100, Train metric: 115.7612
Estimator 27/100, Train metric: 112.9345
Estimator 28/100, Train metric: 110.2707
Estimator 29/100, Train metric: 107.8487
Estimator 30/100, Train metric: 105.6156
Estimator 31/100, Train metric: 103.4932
Estimator 32/100, Train metric: 101.3059
Estimator 33/100, Train metric: 99.2535
Estimator 34/100, Train metric: 97.3164
Estimator 35/100, Train metric: 95.4440
Estimator 36/100, Train metric: 93.6803
Estimator 37/100, Train metric: 92.0293


Estimator 38/100, Train metric: 90.4265
Estimator 39/100, Train metric: 88.8943
Estimator 40/100, Train metric: 87.4100
Estimator 41/100, Train metric: 85.9410
Estimator 42/100, Train metric: 84.5816
Estimator 43/100, Train metric: 83.2309
Estimator 44/100, Train metric: 81.9256
Estimator 45/100, Train metric: 80.6894
Estimator 46/100, Train metric: 79.4676
Estimator 47/100, Train metric: 78.3253
Estimator 48/100, Train metric: 77.2435
Estimator 49/100, Train metric: 76.0672
Estimator 50/100, Train metric: 75.0705
Estimator 51/100, Train metric: 74.0627


Estimator 52/100, Train metric: 73.1462
Estimator 53/100, Train metric: 72.1820
Estimator 54/100, Train metric: 71.2539
Estimator 55/100, Train metric: 70.3354
Estimator 56/100, Train metric: 69.4053
Estimator 57/100, Train metric: 68.5664
Estimator 58/100, Train metric: 67.7062
Estimator 59/100, Train metric: 66.9193
Estimator 60/100, Train metric: 66.0859
Estimator 61/100, Train metric: 65.3456
Estimator 62/100, Train metric: 64.6331
Estimator 63/100, Train metric: 63.9526
Estimator 64/100, Train metric: 63.2129
Estimator 65/100, Train metric: 62.5303


Estimator 66/100, Train metric: 61.9184
Estimator 67/100, Train metric: 61.2744
Estimator 68/100, Train metric: 60.6432
Estimator 69/100, Train metric: 60.0452
Estimator 70/100, Train metric: 59.5317
Estimator 71/100, Train metric: 59.0469
Estimator 72/100, Train metric: 58.5170
Estimator 73/100, Train metric: 58.0346
Estimator 74/100, Train metric: 57.5724
Estimator 75/100, Train metric: 57.1575
Estimator 76/100, Train metric: 56.6305
Estimator 77/100, Train metric: 56.1590
Estimator 78/100, Train metric: 55.7049
Estimator 79/100, Train metric: 55.2801


Estimator 80/100, Train metric: 54.8532
Estimator 81/100, Train metric: 54.4494
Estimator 82/100, Train metric: 54.0603
Estimator 83/100, Train metric: 53.6521
Estimator 84/100, Train metric: 53.2814
Estimator 85/100, Train metric: 52.9341
Estimator 86/100, Train metric: 52.4862
Estimator 87/100, Train metric: 52.1789
Estimator 88/100, Train metric: 51.8715
Estimator 89/100, Train metric: 51.5680
Estimator 90/100, Train metric: 51.2845
Estimator 91/100, Train metric: 50.9917
Estimator 92/100, Train metric: 50.7224
Estimator 93/100, Train metric: 50.3012


Estimator 94/100, Train metric: 49.9954
Estimator 95/100, Train metric: 49.6945
Estimator 96/100, Train metric: 49.4208
Estimator 97/100, Train metric: 49.1523
Estimator 98/100, Train metric: 48.9020
Estimator 99/100, Train metric: 48.5926
         0.05         0.1        0.15         0.2         0.3         0.4  \
0  140.626358  143.288818  144.872330  146.253479  148.180573  149.497971   
1  264.938293  267.162933  269.403717  270.545013  272.533875  274.028900   
2  180.257904  186.223389  188.063004  189.116211  192.819763  194.191483   
3  122.505318  127.292473  128.687988  129.347137  131.210388  133.136536   
4  138.027130  140.707199  144.142487  145.669617  148.499725  149.927521   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0  151.888443  152.723969  154.189377  156.411072  157.309250  158.753342   
1  275.184662  276.693756  278.576080  280.801453  283.383667  285.557068   
2  195.645233  197.750656  199.559921  201.822876  203.164017  204

In [70]:
quantile_regression_print(predictions_PGBM_df,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/PGBM.xlsx")

First few actual values: [160 190 180 110 200]
Coverage of 90% prediction interval: 13.54%


In [71]:
evaluation_features_print(predictions_PGBM_df,x_test,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/PGBM.xlsx")

In [72]:
evaluate_uncertainity_matrix(predictions_PGBM_df, quantiles, "./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/PGBM.xlsx", model_name='PGBM')

Quantile crossing before sorting: 0 of 96 rows (0.0%)


{'empirical_quantiles': [np.float64(0.4270833333333333),
  np.float64(0.4375),
  np.float64(0.4479166666666667),
  np.float64(0.4791666666666667),
  np.float64(0.4791666666666667),
  np.float64(0.4895833333333333),
  np.float64(0.5104166666666666),
  np.float64(0.5104166666666666),
  np.float64(0.53125),
  np.float64(0.53125),
  np.float64(0.5416666666666666),
  np.float64(0.5416666666666666),
  np.float64(0.5625)],
 'average_interval_lengths': [np.float64(9.185238440831503),
  np.float64(15.285039981206259),
  np.float64(21.435458262761433)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(148.34951111806694),
 'average_crps': np.float64(71.15028604328633),
 'nll_values': array([   3.63350097,   60.77709246,    4.98302355,   10.31537883,
          29.34464215,   67.71476473,   98.07926022,   18.16575856,
          75.95726432,   33.44560201,   11.96135798,   54.68462348,
         126.84874147,   70.70283314,  145.70741887,  286.39205759,
         221.39831732,   18.663

In [73]:
calibrate_and_plot_intervals(predictions_PGBM_df, "./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/PGBM.xlsx")

The empirical coverage before calibration is: 16.67%
The empirical coverage after calibration is: 95.83%


In [74]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_PGBM_df,"./drive/MyDrive/Regression/streamflow_uncertainty_analysis/quantile_regression/PGBM.xlsx")